In [0]:


spark.conf.set("fs.azure.account.auth.type.nyctaxisid.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.nyctaxisid.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.nyctaxisid.dfs.core.windows.net", "7c4b429b-ed6b-43f3-a**************0b929d")
spark.conf.set("fs.azure.account.oauth2.client.secret.nyctaxisid.dfs.core.windows.net", "vwA8Q~DhVrjdj*************")
spark.conf.set("fs.azure.account.oauth2.client.endpoint.nyctaxisid.dfs.core.windows.net", "https://login.microsoftonline.com/36c50713-82*****10-48b0-a************/oauth2/token")

In [0]:
dbutils.fs.ls("abfss://bronze@nyctaxisid.dfs.core.windows.net")

[FileInfo(path='abfss://bronze@nyctaxisid.dfs.core.windows.net/trip_type/', name='trip_type/', size=0, modificationTime=1794662872000),
 FileInfo(path='abfss://bronze@nyctaxisid.dfs.core.windows.net/trip_zone/', name='trip_zone/', size=0, modificationTime=1794662880000),
 FileInfo(path='abfss://bronze@nyctaxisid.dfs.core.windows.net/trips2023/', name='trips2023/', size=0, modificationTime=1794665169000)]

#DATA READING

Importing lib

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *


#Reading CSV data

Trip Type Data

In [0]:
df_trip_type = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("abfss://bronze@nyctaxisid.dfs.core.windows.net/trip_type")


In [0]:
df_trip_type.display()

trip_type,description
1,Street-hail
2,Dispatch


Reading Trip Zone

In [0]:
df_trip_zone = spark.read.format("csv")\
                    .option("header", "true")\
                     .option("inferSchema", "true")\
                     .load("abfss://bronze@nyctaxisid.dfs.core.windows.net/trip_zone")

In [0]:
df_trip_zone.display()

LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


TRIP DATA

In [0]:
df_trip = spark.read.format("parquet") \
    .option("header", "true") \
    .schema(my_schema) \
    .option("recursiveFileLookup", "true") \
    .load("abfss://bronze@nyctaxisid.dfs.core.windows.net/trips2023")

In [0]:
print(type(df_trip))

<class 'pyspark.sql.connect.dataframe.DataFrame'>


modifying my schema 

In [0]:
my_schema = '''
    VendorID BIGINT,
    lpep_pickup_datetime TIMESTAMP,
    lpep_dropoff_datetime TIMESTAMP,
    store_and_fwd_flag STRING,
    RatecodeID BIGINT,
    PULocationID BIGINT,
    DOLocationID BIGINT,
    passenger_count BIGINT,
    trip_distance DOUBLE,
    fare_amount DOUBLE,
    extra DOUBLE,
    mta_tax DOUBLE,
    tip_amount DOUBLE,
    tolls_amount DOUBLE,
    ehail_fee DOUBLE,
    improvement_surcharge DOUBLE,
    total_amount DOUBLE,
    payment_type BIGINT,
    trip_type BIGINT,
    congestion_surcharge DOUBLE
'''

In [0]:
df_trip.display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,196,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,74,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,229,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,235,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,160,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,264,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,41,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,7,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0


## DATA TRANSFORMATIONS

In [0]:
df_trip_type.display()


trip_type,description
1,Street-hail
2,Dispatch


In [0]:
df_trip_type = df_trip_type.withColumnRenamed('description', 'Trip_Description')
df_trip_type.display()


trip_type,Trip_Description
1,Street-hail
2,Dispatch


WRITING TRIP TYPE

In [0]:
df_trip_type.write.format("parquet")\
                  .mode("append")\
                  .option("path", "abfss://silver@nyctaxisid.dfs.core.windows.net/trip_type")\
                  .save()

In [0]:
df_trip_zone.display()


LocationID,Borough,Zone,service_zone
1,EWR,Newark Airport,EWR
2,Queens,Jamaica Bay,Boro Zone
3,Bronx,Allerton/Pelham Gardens,Boro Zone
4,Manhattan,Alphabet City,Yellow Zone
5,Staten Island,Arden Heights,Boro Zone
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone
7,Queens,Astoria,Boro Zone
8,Queens,Astoria Park,Boro Zone
9,Queens,Auburndale,Boro Zone
10,Queens,Baisley Park,Boro Zone


SPLITTING THE ZONE NAME USING SPLIT()


In [0]:
df_trip_zone = df_trip_zone.withColumn("ZONE 1", split(col("Zone"), "/")[0])\
                           .withColumn("ZONE 2", split(col("Zone"), "/")[1])\
                            .withColumn("Zone 3", split(col("Zone"), "/")[2])
                           
df_trip_zone.display()

                           


LocationID,Borough,Zone,service_zone,ZONE 1,ZONE 2,Zone 3
1,EWR,Newark Airport,EWR,Newark Airport,null,null
2,Queens,Jamaica Bay,Boro Zone,Jamaica Bay,null,null
3,Bronx,Allerton/Pelham Gardens,Boro Zone,Allerton,Pelham Gardens,null
4,Manhattan,Alphabet City,Yellow Zone,Alphabet City,null,null
5,Staten Island,Arden Heights,Boro Zone,Arden Heights,null,null
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,Arrochar,Fort Wadsworth,null
7,Queens,Astoria,Boro Zone,Astoria,null,null
8,Queens,Astoria Park,Boro Zone,Astoria Park,null,null
9,Queens,Auburndale,Boro Zone,Auburndale,null,null
10,Queens,Baisley Park,Boro Zone,Baisley Park,null,null


NOW WRITING TRIP ZONE DATA TO SILVER

In [0]:
df_trip_zone.write.format("parquet")\
                  .mode("append")\
                  .option("path", "abfss://silver@nyctaxisid.dfs.core.windows.net/trip_zone")\
                  .save()

**TRIP DATA (12 MONTHS)**

In [0]:
df_trip.display()

VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,196,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,74,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,229,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,235,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,160,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,264,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,41,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,7,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0


In [0]:
df_trip = df_trip.withColumn("Trip_Date", to_date(col("lpep_pickup_datetime")))\
                 .withColumn("Trip_Year", year(col("lpep_pickup_datetime")))\
                 .withColumn("Trip_Month", month(col("lpep_pickup_datetime")))\
                     .withColumn("Hours", hour(col("lpep_pickup_datetime")))
df_trip.display()


VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,Trip_Date,Trip_Year,Trip_Month,Hours
2,2023-03-01T00:25:10Z,2023-03-01T00:35:47Z,N,1,82,196,1,2.36,13.5,1.0,0.5,0.0,0.0,null,1.0,16.0,2,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,-6.5,-1.0,-0.5,0.0,0.0,null,-1.0,-9.0,3,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:14:29Z,2023-03-01T00:25:04Z,N,1,7,7,1,0.78,6.5,1.0,0.5,0.0,0.0,null,1.0,9.0,3,1,0.0,2023-03-01,2023,3,0
2,2023-02-28T22:59:46Z,2023-02-28T23:08:38Z,N,1,166,74,1,1.66,11.4,1.0,0.5,2.78,0.0,null,1.0,16.68,1,1,0.0,2023-02-28,2023,2,22
2,2023-03-01T00:54:03Z,2023-03-01T01:03:14Z,N,1,236,229,1,3.14,15.6,1.0,0.5,4.17,0.0,null,1.0,25.02,1,1,2.75,2023-03-01,2023,3,0
2,2023-03-01T01:00:09Z,2023-03-01T01:14:37Z,N,1,75,235,1,5.69,23.3,1.0,0.5,4.0,0.0,null,1.0,29.8,1,1,0.0,2023-03-01,2023,3,1
2,2023-03-01T00:09:45Z,2023-03-01T00:26:06Z,N,1,260,160,1,2.92,17.7,1.0,0.5,4.04,0.0,null,1.0,24.24,1,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:39:30Z,2023-03-01T00:39:33Z,N,5,95,264,5,0.0,35.0,0.0,0.0,1.0,0.0,null,1.0,37.0,1,2,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:03:07Z,2023-03-01T00:14:44Z,N,1,244,41,1,3.34,16.3,1.0,0.5,5.64,0.0,null,1.0,24.44,1,1,0.0,2023-03-01,2023,3,0
2,2023-03-01T00:42:56Z,2023-03-01T00:49:57Z,N,1,83,7,5,1.75,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0,2023-03-01,2023,3,0


In [0]:
print(df_trip.columns)


['VendorID', 'lpep_pickup_datetime', 'lpep_dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'ehail_fee', 'improvement_surcharge', 'total_amount', 'payment_type', 'trip_type', 'congestion_surcharge', 'Trip_Date', 'Trip_Year', 'Trip_Month', 'Hours']


WRITING TRIP TO SILVER

In [0]:
from functools import reduce

# Source files have inconsistent column types (e.g. DOUBLE vs BIGINT), so read each file and cast to my_schema
target_schema = spark.createDataFrame([], my_schema).schema
trip_files = [f.path for f in dbutils.fs.ls("abfss://bronze@nyctaxisid.dfs.core.windows.net/trips2023/trip-data")]
df_trip = reduce(lambda a, b: a.unionByName(b), [
    spark.read.parquet(p).select([col(c.name).cast(c.dataType).alias(c.name) for c in target_schema])
    for p in trip_files
])
df_trip = df_trip.withColumn("Trip_Date", to_date(col("lpep_pickup_datetime")))\
                 .withColumn("Trip_Year", year(col("lpep_pickup_datetime")))\
                 .withColumn("Trip_Month", month(col("lpep_pickup_datetime")))\
                 .withColumn("Hours", hour(col("lpep_pickup_datetime")))

df_trip.write.format("parquet")\
                  .mode("overwrite")\
                  .option("path", "abfss://silver@nyctaxisid.dfs.core.windows.net/trips2023")\
                  .save()

ANALYSIS

In [0]:
display(df_trip)


VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,Trip_Date,Trip_Year,Trip_Month,Hours
2,2023-01-01T00:26:10Z,2023-01-01T00:37:11Z,N,1,166,143,1,2.58,14.9,1.0,0.5,4.03,0.0,null,1.0,24.18,1,1,2.75,2023-01-01,2023,1,0
2,2023-01-01T00:51:03Z,2023-01-01T00:57:49Z,N,1,24,43,1,1.81,10.7,1.0,0.5,2.64,0.0,null,1.0,15.84,1,1,0.0,2023-01-01,2023,1,0
2,2023-01-01T00:35:12Z,2023-01-01T00:41:32Z,N,1,223,179,1,0.0,7.2,1.0,0.5,1.94,0.0,null,1.0,11.64,1,1,0.0,2023-01-01,2023,1,0
1,2023-01-01T00:13:14Z,2023-01-01T00:19:03Z,N,1,41,238,1,1.3,6.5,0.5,1.5,1.7,0.0,null,1.0,10.2,1,1,0.0,2023-01-01,2023,1,0
1,2023-01-01T00:33:04Z,2023-01-01T00:39:02Z,N,1,41,74,1,1.1,6.0,0.5,1.5,0.0,0.0,null,1.0,8.0,1,1,0.0,2023-01-01,2023,1,0
2,2023-01-01T00:53:31Z,2023-01-01T01:11:04Z,N,1,41,262,1,2.78,17.7,1.0,0.5,0.0,0.0,null,1.0,22.95,2,1,2.75,2023-01-01,2023,1,0
1,2023-01-01T00:09:14Z,2023-01-01T00:26:39Z,N,1,181,45,2,3.8,19.1,3.75,1.5,4.85,0.0,null,1.0,29.2,1,1,2.75,2023-01-01,2023,1,0
2,2023-01-01T00:11:58Z,2023-01-01T00:24:55Z,N,1,24,75,1,1.88,14.2,1.0,0.5,0.0,0.0,null,1.0,16.7,2,1,0.0,2023-01-01,2023,1,0
2,2023-01-01T00:41:29Z,2023-01-01T00:46:26Z,N,1,41,166,2,1.11,7.2,1.0,0.5,1.0,0.0,null,1.0,10.7,1,1,0.0,2023-01-01,2023,1,0
2,2023-01-01T00:50:32Z,2023-01-01T01:13:42Z,N,1,24,140,1,4.22,24.7,1.0,0.5,3.0,0.0,null,1.0,32.95,1,1,2.75,2023-01-01,2023,1,0


Databricks visualization. Run in Databricks to view.